## 🔍 Stop Loss Logic Verification Summary

### ✅ Correct Implementation (Both cells now fixed)

**Key components of proper stop loss logic:**

1. **Trading Costs Applied:**
   - Entry: `entry_price = price * (1 + trading_cost_pct)` 
   - Exit (normal sell): `exit_price = price * (1 - trading_cost_pct)`
   - Exit (stop loss): `exit_price = stop_loss_price * (1 - trading_cost_pct)` ← **Caps loss at stop level**

2. **Stop Loss Priority (checked FIRST):**
   ```python
   if price <= stop_loss_price:  # Priority 1: Stop loss - exits at stop_loss_price
       exit_price = stop_loss_price * (1 - trading_cost_pct)  # Not current price!
   elif prob <= sell_threshold:  # Priority 2: Sell signal
       exit_price = price * (1 - trading_cost_pct)
   ```

3. **Percentage-Based Returns:**
   - ✅ `balance *= (exit_price / entry_price)` - Correct
   - ❌ `balance += (sell_price - buy_price)` - WRONG (was in old code)

4. **Stop Loss Calculation:**
   - `stop_loss_price = entry_price * (1 - stop_loss_pct)`
   - For 2.5% stop loss with 0.1% trading cost: max loss ≈ -2.6%

### 📊 Both Cells Are Now Consistent:
- **Cell 9**: Simple strategy execution with detailed trade history
- **Cell 10**: Threshold optimization with stop loss (grid search)

## 15-Minute Trading Strategy Backtest

Load models and test them with backtesting based on balance and risk management techniques

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
import talib as tal
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Try to import TensorFlow/Keras for neural network models
try:
    import tensorflow as tf
    from tensorflow import keras
    KERAS_AVAILABLE = True
    print("✅ TensorFlow/Keras available")
except ImportError:
    KERAS_AVAILABLE = False
    print("⚠️ TensorFlow/Keras not available - only sklearn models supported")

print("✅ Libraries imported successfully")

✅ TensorFlow/Keras available
✅ Libraries imported successfully


In [2]:
# ============================================================================
# LOAD ALL 15m MODELS FROM RESULTS FOLDER
# ============================================================================
import os
import json
import pickle
import pandas as pd

# Path to the results root folder
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"

def load_all_15m_models():
    """Load all models from the 15m results folder."""
    models_15m = {}

    # Find the 15m results folder
    results_15m_path = None
    for folder_name in os.listdir(RESULTS_ROOT):
        if folder_name.startswith('ETH_USD_15m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
            results_15m_path = os.path.join(RESULTS_ROOT, folder_name)
            break

    if results_15m_path is None:
        print("❌ No 15m results folder found!")
        return models_15m

    print(f"📂 Found 15m results folder: {os.path.basename(results_15m_path)}")

    # Load config and results
    config_path = os.path.join(results_15m_path, "config.json")
    results_path = os.path.join(results_15m_path, "results_summary.json")
    registry_path = os.path.join(results_15m_path, "models_registry.json")
    features_path = os.path.join(results_15m_path, "features.json")

    if not os.path.exists(config_path) or not os.path.exists(results_path):
        print("❌ Missing config or results files!")
        return models_15m

    # Load metadata
    config = json.load(open(config_path))
    results = json.load(open(results_path))

    models_registry = None
    if os.path.exists(registry_path):
        try:
            models_registry = json.load(open(registry_path))
        except:
            models_registry = None

    features_config = None
    if os.path.exists(features_path):
        try:
            features_config = json.load(open(features_path))
        except:
            features_config = None

    # Get all models from ranking
    ranking = results.get('ranking', [])
    all_models_info = results.get('all_models', {})

    print(f"🔍 Found {len(ranking)} models in 15m results")
    print(f"🏆 Best model: {results.get('best_model', 'N/A')}")

    # Load each model
    models_dir = os.path.join(results_15m_path, "models")
    loaded_count = 0

    for model_info in ranking:
        model_name = model_info['model']

        # Find model file
        model_path = None
        model_type = None

        # Check models directory first
        if os.path.exists(models_dir):
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(models_dir, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        # Check root directory if not found
        if model_path is None:
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(results_15m_path, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        if model_path is None:
            print(f"⚠️ Model file not found for: {model_name}")
            continue

        # Load the model
        try:
            if model_type == 'keras' and KERAS_AVAILABLE:
                model = keras.models.load_model(model_path)
            elif model_type in ['pkl', 'sklearn']:
                with open(model_path, 'rb') as f:
                    model = pickle.load(f)
            else:
                print(f"⚠️ Unsupported model type {model_type} for: {model_name}")
                continue

            # Get additional info
            model_class = None
            if models_registry:
                model_entry = models_registry.get('models', {}).get(model_name, {})
                model_class = model_entry.get('model_class')

            # Store model info
            models_15m[model_name] = {
                'model': model,
                'model_type': model_type,
                'model_class': model_class,
                'model_path': model_path,
                'feature_set': model_info.get('feature_set', 'X'),
                'train_acc': model_info.get('train_acc', 0),
                'forward_acc': model_info.get('forward_acc', 0),
                'rank': model_info.get('rank', 999),
                'config': config,
                'features_config': features_config
            }

            loaded_count += 1
            print(f"✅ Loaded {model_name} ({model_type}) - Rank: {model_info.get('rank', 'N/A')}, Acc: {model_info.get('forward_acc', 0):.4f}")

        except Exception as e:
            print(f"❌ Failed to load {model_name}: {e}")
            continue

    print(f"\n📊 Successfully loaded {loaded_count} out of {len(ranking)} 15m models")
    return models_15m

# ============================================================================
# EXECUTE LOADING
# ============================================================================
print("=" * 70)
print("🚀 LOADING ALL 15m MODELS")
print("=" * 70)

MODELS_15m = load_all_15m_models()

if len(MODELS_15m) == 0:
    print("❌ No 15m models were loaded!")
else:
    print(f"\n📈 Available 15m models: {list(MODELS_15m.keys())}")

    # Show summary
    df_15m_models = pd.DataFrame([
        {
            'Model': name,
            'Type': info['model_type'],
            'Class': info['model_class'] or 'N/A',
            'Rank': info['rank'],
            'Train_Acc': f"{info['train_acc']:.4f}",
            'Forward_Acc': f"{info['forward_acc']:.4f}",
            'Feature_Set': info['feature_set']
        }
        for name, info in MODELS_15m.items()
    ]).sort_values('Rank')

    print("\n🏆 15m MODELS SUMMARY (sorted by rank):")
    display(df_15m_models)

print("=" * 70)

🚀 LOADING ALL 15m MODELS
📂 Found 15m results folder: ETH_USD_15m_20260115_162925
🔍 Found 30 models in 15m results
🏆 Best model: AdaBoost_Full
✅ Loaded AdaBoost_Full (pkl) - Rank: 1, Acc: 0.5394
✅ Loaded LSTM_Full (keras) - Rank: 2, Acc: 0.5377
✅ Loaded LSTM_Attention_Full (keras) - Rank: 3, Acc: 0.5370
✅ Loaded LSTM_X (keras) - Rank: 4, Acc: 0.5357
✅ Loaded LSTM_Attention_X (keras) - Rank: 5, Acc: 0.5351
✅ Loaded AdaBoost_X (pkl) - Rank: 6, Acc: 0.5350
✅ Loaded RandomForest_Full (pkl) - Rank: 7, Acc: 0.5347
✅ Loaded CNN_X (keras) - Rank: 8, Acc: 0.5326
✅ Loaded NN_Full (keras) - Rank: 9, Acc: 0.5304
✅ Loaded RandomForest_X (pkl) - Rank: 10, Acc: 0.5298
✅ Loaded Lion_X (keras) - Rank: 11, Acc: 0.5287
✅ Loaded LogisticRegression_ElasticNet_X (pkl) - Rank: 12, Acc: 0.5281
✅ Loaded LinearSVC_X (pkl) - Rank: 13, Acc: 0.5270
✅ Loaded GRU_X (keras) - Rank: 14, Acc: 0.5261
✅ Loaded LogisticRegression_ElasticNet_Full (pkl) - Rank: 15, Acc: 0.5251
✅ Loaded CatBoost_Full (pkl) - Rank: 16, Acc: 0.

,Model,Type,Class,Rank,Train_Acc,Forward_Acc,Feature_Set
0,AdaBoost_Full,pkl,AdaBoostClassifier,1,0.5455,0.5394,Full
1,LSTM_Full,keras,Sequential,2,0.5317,0.5377,Full
2,LSTM_Attention_Full,keras,Functional,3,0.5393,0.5370,Full
3,LSTM_X,keras,Sequential,4,0.5415,0.5357,X
4,LSTM_Attention_X,keras,Functional,5,0.5290,0.5351,X
5,AdaBoost_X,pkl,AdaBoostClassifier,6,0.5435,0.5350,X
6,RandomForest_Full,pkl,RandomForestClassifier,7,0.5439,0.5347,Full
7,CNN_X,keras,Sequential,8,0.5307,0.5326,X
8,NN_Full,keras,Sequential,9,0.5388,0.5304,Full
9,RandomForest_X,pkl,RandomForestClassifier,10,0.5458,0.5298,X


In [ ]:
# ============================================================================
# LOAD SCALER VALUES FROM CONFIG
# ============================================================================
# Load scaler values from config
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"
RESULTS_15m_FOLDER = None
for folder_name in os.listdir(RESULTS_ROOT):
    if folder_name.startswith('ETH_USD_15m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
        RESULTS_15m_FOLDER = os.path.join(RESULTS_ROOT, folder_name)
        break

if RESULTS_15m_FOLDER:
    with open(os.path.join(RESULTS_15m_FOLDER, "config.json"), "r") as f:
        config_15m = json.load(f)
    
    scaler_15m = config_15m["scaler"]
    print(f"✅ Config loaded from: {os.path.basename(RESULTS_15m_FOLDER)}")
    print(f"📋 Scaler keys: {list(scaler_15m.keys())}")
    
    x_mean_dict_15m = scaler_15m["X_mean"]
    x_std_dict_15m = scaler_15m["X_std"]
    x_full_mean_dict_15m = scaler_15m["X_full_mean"]
    x_full_std_dict_15m = scaler_15m["X_full_std"]
    
    print(f"   - X_mean features: {len(x_mean_dict_15m)}")
    print(f"   - X_std features: {len(x_std_dict_15m)}")
    print(f"   - X_full_mean features: {len(x_full_mean_dict_15m)}")
    print(f"   - X_full_std features: {len(x_full_std_dict_15m)}")
else:
    print("❌ ERROR: No 15m results folder found!")

In [13]:
# ============================================================================

# LOAD 15M DATA FROM YFINANCE

# ============================================================================

import yfinance as yf

import pandas as pd

import numpy as np

import warnings

warnings.filterwarnings('ignore')



SYMBOL_15M = 'ETH-USD'  # Change if needed



print("🚀 Fetching 15m data for:", SYMBOL_15M)



try:

    ticker = yf.Ticker(SYMBOL_15M)



    # Primary: max period

    df_15m = ticker.history(period='max', interval='15m', prepost=True, actions=False)



    # Fallbacks if empty

    if df_15m is None or df_15m.empty:

        print("⚠️ primary attempt empty — trying period='60d'...")

        df_15m = ticker.history(period='60d', interval='15m', prepost=True, actions=False)



    if df_15m is None or df_15m.empty:

        print("⚠️ still empty — trying yf.download(period='30d')...")

        df_15m = yf.download(SYMBOL_15M, period='30d', interval='15m', progress=False)



    if df_15m is None or df_15m.empty:

        print("❌ No 15m data retrieved from yfinance.")

        DF_15M_MAX = pd.DataFrame()

    else:

        # Clean

        df_15m = df_15m[~df_15m.index.duplicated(keep='first')]

        df_15m = df_15m.sort_index()

        df_15m = df_15m.dropna(how='all')

        df_15m = df_15m.ffill()  # Fixed: must assign the result



        # Derived columns

        df_15m['Returns'] = df_15m['Close'].pct_change()

        df_15m['Log_Return'] = np.log(df_15m['Close'] / df_15m['Close'].shift(1))



        # Store

        DF_15M_MAX = df_15m.copy()



        print("✅ Data stored in variable: DF_15M_MAX")

        print(f"   Shape: {DF_15M_MAX.shape}")

        print(f"   Date range: {DF_15M_MAX.index[0]} to {DF_15M_MAX.index[-1]}")



        print("\n🔍 FIRST 3 ROWS:")

        display(DF_15M_MAX.head(3))

        print("\n🔍 LAST 3 ROWS:")

        display(DF_15M_MAX.tail(3))



except Exception as e:

    print(f"❌ Error fetching data: {e}")

    import traceback

    traceback.print_exc()

    DF_15M_MAX = pd.DataFrame()


🚀 Fetching 15m data for: ETH-USD
✅ Data stored in variable: DF_15M_MAX
   Shape: (5761, 7)
   Date range: 2025-12-19 08:00:00+00:00 to 2026-02-17 07:54:00+00:00

🔍 FIRST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2025-12-19 08:00:00+00:00,2949.798340,2960.566406,2949.644531,2951.921631,49958912,NaN,NaN
2025-12-19 08:15:00+00:00,2952.803223,2954.374756,2943.144287,2943.144287,311214080,-0.002973,-0.002978
2025-12-19 08:30:00+00:00,2939.667969,2948.130127,2938.382324,2946.515381,488550400,0.001145,0.001145



🔍 LAST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2026-02-17 07:30:00+00:00,1973.617676,1981.970459,1973.617676,1980.781006,142354432,0.003702,0.003695
2026-02-17 07:45:00+00:00,1981.413818,1985.272095,1980.865112,1985.001709,175726592,0.002131,0.002129
2026-02-17 07:54:00+00:00,1985.981079,1985.981079,1985.981079,1985.981079,0,0.000493,0.000493


In [14]:
def calculate_indicators_15m(df):
    """
    Calculate technical indicators for 15m feature set only.
    """
    df = df.copy()

    # Target variable (for reference)
    df["return"] = df["Close"].pct_change()
    df["target"] = (df["return"].shift(-1) > 0).astype(int)

    # Trend Indicators
    df["EMA_20"] = tal.EMA(df["Close"], timeperiod=20)
    df["SAR"] = tal.SAR(df['High'], df['Low'], acceleration=0.02, maximum=0.2)
    df["SMA_10"] = df["Close"].rolling(10).mean()
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    df["STD_10"] = df["Close"].rolling(10).std()
    df["STD_20"] = df["Close"].rolling(20).std()

    # Momentum Indicators
    df["RSI_20"] = tal.RSI(df["Close"], timeperiod=20)
    df["ROC_10"] = df["Close"].pct_change(10)
    df["CCI_20"] = tal.CCI(df['High'], df['Low'], df['Close'], timeperiod=20)
    df["MACD"], df["MACD_signal"], df["MACD_hist"] = tal.MACD(df['Close'])

    # Volatility Indicators
    df["ATR_14"] = tal.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    sma = df["Close"].rolling(20).mean()
    std = df["Close"].rolling(20).std()
    df["BB_upper"] = sma + (std * 2)
    df["BB_lower"] = sma - (std * 2)
    df["BW_20"] = (df["BB_upper"] - df["BB_lower"]) / sma

    # Chaikin Money Flow (CMF)
    def calculate_cmf(high, low, close, volume, period=20):
        """Calculate Chaikin Money Flow"""
        mfv = ((close - low) - (high - close)) / (high - low + 1e-10) * volume
        cmf = mfv.rolling(period).sum() / volume.rolling(period).sum()
        return cmf
    
    df["CMF_20"] = calculate_cmf(df['High'], df['Low'], df['Close'], df['Volume'], period=20)

    # Volume lags
    df["Volume_lag1"] = df["Volume"].shift(1)
    df["Volume_lag2"] = df["Volume"].shift(2)

    # Price-based Indicators
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    # Lag features
    for i in range(1, 6):
        df[f"Lag_{i}"] = df["return"].shift(i)

    # Keep only required 15m feature columns + target/return
    required_cols = [
        "Open",
        "Close",
        "Volume",
        "EMA_20",
        "SAR",
        "RSI_20",
        "ROC_10",
        "CCI_20",
        "MACD",
        "MACD_signal",
        "MACD_hist",
        "ATR_14",
        "BB_lower",
        "BW_20",
        "SMA_10",
        "SMA_20",
        "SMA_50",
        "STD_10",
        "STD_20",
        "CMF_20",
        "Log_Return",
        "Lag_1",
        "Lag_2",
        "Lag_3",
        "Lag_4",
        "Lag_5",
        "Volume_lag1",
        "Volume_lag2",
    ]

    # Drop rows with NaNs in required columns
    df = df.dropna(subset=required_cols + ["target", "return"])

    return df[required_cols + ["target", "return"]]


def _assert_features_match(tf, feature_cols, selected_features, full_features):
    selected_set = set(selected_features or [])
    full_set = set(full_features or [])
    extracted_set = set(feature_cols or [])

    if selected_set and not selected_set.issubset(extracted_set):
        missing = sorted(selected_set - extracted_set)
        raise ValueError(
            f"Feature mismatch for {tf} (selected_features). "
            f"Missing: {missing}"
        )

    if full_set and selected_set and not selected_set.issubset(full_set):
        missing_from_full = sorted(selected_set - full_set)
        raise ValueError(
            f"Feature mismatch for {tf} (full_features). "
            f"Selected not in full_features: {missing_from_full}"
        )


# Define feature sets for 15m (from your config)
X_SELECTED_FEATURES_15m = [
    "Open",
    "Volume",
    "RSI_20",
    "ROC_10",
    "CCI_20",
    "MACD",
    "MACD_hist",
    "ATR_14",
    "BW_20",
    "CMF_20",
    "Log_Return",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_4",
    "Lag_5",
    "Volume_lag1",
    "Volume_lag2",
]

X_FULL_FEATURES_15m = [
    "Open",
    "Close",
    "Volume",
    "EMA_20",
    "SAR",
    "RSI_20",
    "ROC_10",
    "CCI_20",
    "MACD",
    "MACD_signal",
    "MACD_hist",
    "ATR_14",
    "BB_lower",
    "BW_20",
    "SMA_10",
    "SMA_20",
    "SMA_50",
    "STD_10",
    "STD_20",
    "CMF_20",
    "Log_Return",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_4",
    "Lag_5",
    "Volume_lag1",
    "Volume_lag2",
]

# Prepare features for 15m timeframe using already-loaded 15m data
processed_by_timeframe = {}
feature_cols_by_timeframe = {}
X_by_timeframe = {}

if 'DF_15M_MAX' in globals() and isinstance(DF_15M_MAX, pd.DataFrame) and not DF_15M_MAX.empty:
    df_15m_source = DF_15M_MAX.copy()
    print("Using DF_15M_MAX as source for 15m features")
else:
    raise ValueError("No loaded 15m data found. Please run the 15m yfinance loader cell first.")

tf = "15m"

df_proc = calculate_indicators_15m(df_15m_source)
processed_by_timeframe[tf] = df_proc

selected_feats = X_SELECTED_FEATURES_15m if 'X_SELECTED_FEATURES_15m' in globals() else []
full_feats = X_FULL_FEATURES_15m if 'X_FULL_FEATURES_15m' in globals() else []

# Use full feature set instead of selected features
feature_cols_tf = full_feats if full_feats else [
    col for col in df_proc.columns if col not in ["target", "return"]
]

_assert_features_match(
    tf,
    df_proc.columns.tolist(),
    selected_feats,
    full_feats
)

X_tf = df_proc[feature_cols_tf].copy()

feature_cols_by_timeframe[tf] = feature_cols_tf
X_by_timeframe[tf] = X_tf

tf_safe = tf.replace("-", "_").replace(".", "_")
globals()[f"DF_PROCESSED_{tf_safe}"] = df_proc
globals()[f"FEATURE_COLS_{tf_safe}"] = feature_cols_tf
globals()[f"X_{tf_safe}"] = X_tf

print(f"✅ {tf}: processed={df_proc.shape} | features={len(feature_cols_tf)} | X={X_tf.shape}")
print(f"\n📋 Feature Configuration:")
print(f"  - X_selected_features: {len(X_SELECTED_FEATURES_15m)} features")
print(f"  - X_full_features: {len(X_FULL_FEATURES_15m)} features")

df_processed = df_proc
feature_cols = feature_cols_tf
X = X_tf
TIMEFRAME = tf
print(f"\n✅ Active timeframe ready: {TIMEFRAME}")

Using DF_15M_MAX as source for 15m features
✅ 15m: processed=(5712, 30) | features=28 | X=(5712, 28)

📋 Feature Configuration:
  - X_selected_features: 18 features
  - X_full_features: 28 features

✅ Active timeframe ready: 15m


In [ ]:
# ============================================================================
# BUILD ENSEMBLE PREDICTIONS FROM TOP 3 MODELS
# ============================================================================
import pandas as pd
import numpy as np

print("="*70)
print("🚀 BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS")
print("="*70)

# Select top 3 models by forward_acc
if len(MODELS_15m) == 0:
    raise ValueError("No models loaded. Run model loading cell first.")

top_3_models = sorted(
    [(name, info) for name, info in MODELS_15m.items()],
    key=lambda x: x[1]['forward_acc'],
    reverse=True
)[:3]

print(f"\n📊 Top 3 models selected by forward accuracy:")
for i, (name, info) in enumerate(top_3_models, 1):
    print(f"   {i}. {name} - Acc: {info['forward_acc']:.4f}, Features: {info['feature_set']}")

# Prepare features (full set, 28 features)
full_features_list = X_FULL_FEATURES_15m if 'X_FULL_FEATURES_15m' in globals() else []
selected_features_list = X_SELECTED_FEATURES_15m if 'X_SELECTED_FEATURES_15m' in globals() else []

# Create feature matrices
if not DF_PROCESSED_15m.empty:
    X_full = DF_PROCESSED_15m[full_features_list].copy()
    X_selected = DF_PROCESSED_15m[selected_features_list].copy()
    
    # Drop NaNs separately for each feature set
    X_full = X_full.dropna()
    X_selected = X_selected.dropna()
    
    # Find common index across both
    common_index = X_full.index.intersection(X_selected.index)
    
    X_full = X_full.loc[common_index]
    X_selected = X_selected.loc[common_index]
    
    print(f"\n📋 Feature matrices created:")
    print(f"   - X_full (28 features): {X_full.shape}")
    print(f"   - X_selected (18 features): {X_selected.shape}")
    print(f"   - Common index: {len(common_index)} rows")
else:
    raise ValueError("DF_PROCESSED_15m is empty. Run indicators cell first.")

# Generate predictions from each model
model_predictions = {}
model_features_used = []
TOP_3_MODEL_NAMES = []
MODEL_FEATURES_USED = []

for model_name, model_info in top_3_models:
    model = model_info['model']
    
    # Determine feature requirement
    try:
        n_features_expected = model.n_features_in_
    except:
        # Try alternative attributes
        try:
            n_features_expected = len(model.feature_names_in_)
        except:
            n_features_expected = None
    
    # Select feature set based on model requirement
    if n_features_expected:
        if n_features_expected == len(selected_features_list):
            X_model = X_selected.copy()
            feature_type = f"selected ({len(selected_features_list)} features)"
        elif n_features_expected == len(full_features_list):
            X_model = X_full.copy()
            feature_type = f"full ({len(full_features_list)} features)"
        else:
            # Default to full features if mismatch
            X_model = X_full.copy()
            feature_type = f"full ({len(full_features_list)} features)"
    else:
        # Default to full features
        X_model = X_full.copy()
        feature_type = f"full ({len(full_features_list)} features)"
    
    # Determine which scaler to use based on feature set
    feature_set = model_info['feature_set']
    if feature_set == 'Full' or n_features_expected == len(full_features_list):
        mean_dict = x_full_mean_dict_15m
        std_dict = x_full_std_dict_15m
    else:
        mean_dict = x_mean_dict_15m
        std_dict = x_std_dict_15m

    # Z-score normalize using config values
    mean_s = pd.Series(mean_dict).reindex(X_model.columns).astype(float)
    std_s = pd.Series(std_dict).reindex(X_model.columns).astype(float).replace(0, np.nan)
    X_z = (X_model - mean_s) / std_s
    X_z = X_z.replace([np.inf, -np.inf], np.nan).dropna()
    
    # Generate predictions for this model
    try:
        # Check if this is an LSTM/Neural Network model
        is_lstm = False
        if KERAS_AVAILABLE and hasattr(model, 'predict'):
            # Check if it's a Keras model by looking for typical Keras attributes
            if hasattr(model, 'input_shape') or str(type(model).__module__).startswith('tensorflow'):
                is_lstm = True
                print(f"\n🔹 {model_name}: Detected as LSTM/Neural Network model")
        
        if is_lstm and KERAS_AVAILABLE:
            try:
                # For LSTM models: reshape to 3D (samples, 1, features) - single timestep
                X_z_reshaped = X_z.values.reshape((X_z.shape[0], 1, X_z.shape[1]))
                print(f"   - Reshaping for LSTM: {X_z.shape} → {X_z_reshaped.shape}")
                
                # Try to build the model with correct input shape if it has unknown rank
                if hasattr(model, 'built') and not model.built:
                    print(f"   - Building LSTM model with input shape: {X_z_reshaped.shape}")
                    model.build(input_shape=(None, X_z_reshaped.shape[1], X_z_reshaped.shape[2]))
                
                # Get predictions with small batch size to avoid memory issues
                proba = model.predict(X_z_reshaped, verbose=0, batch_size=128)
                
                # Handle different output shapes
                if len(proba.shape) > 1 and proba.shape[1] > 1:
                    prob_up_model = proba[:, 1]  # Binary classification: class 1 probability
                else:
                    prob_up_model = proba.flatten()  # Already probability for up class
                
                print(f"   - LSTM prediction successful: {len(prob_up_model)} predictions")
            except Exception as lstm_error:
                print(f"   ⚠️ LSTM prediction failed: {str(lstm_error)}")
                print(f"   - Attempting fallback flatten prediction...")
                try:
                    # Flatten and try 2D prediction
                    X_z_2d = X_z.values
                    proba = model.predict(X_z_2d, verbose=0, batch_size=128)
                    prob_up_model = proba.flatten() if len(proba.shape) > 1 else proba
                except:
                    # Skip this model if both attempts fail
                    print(f"   - Skipping {model_name} due to prediction errors")
                    raise lstm_error
        else:
            # Standard sklearn models
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_z)
                prob_up_model = proba[:, 1]
            else:
                # Fallback: use decision function or predictions
                if hasattr(model, 'decision_function'):
                    scores = model.decision_function(X_z)
                    prob_up_model = 1 / (1 + np.exp(-scores))
                else:
                    preds = model.predict(X_z)
                    prob_up_model = preds.astype(float)
        
        # Clip to valid range
        prob_up_model = np.clip(prob_up_model, 0, 1)
        
        # Store as pandas Series with index for proper alignment
        model_predictions[model_name] = pd.Series(prob_up_model, index=X_model.index)
        model_features_used.append(feature_type)
        TOP_3_MODEL_NAMES.append(model_name)
        MODEL_FEATURES_USED.append(feature_type)
        
        print(f"✅ {model_name}: {feature_type} | predictions: {len(prob_up_model)} | mean prob: {prob_up_model.mean():.4f}")
        
    except Exception as e:
        print(f"❌ {model_name}: Prediction failed - {str(e)}")
        print(f"   - Model type: {type(model)}")
        print(f"   - Input shape: {X_z.shape}")
        import traceback
        traceback.print_exc()
        continue

# Align predictions across models
if len(model_predictions) == 0:
    raise ValueError("No successful model predictions generated!")

# Find common index across all predictions
common_pred_index = list(model_predictions.values())[0].index
for pred_series in list(model_predictions.values())[1:]:
    common_pred_index = common_pred_index.intersection(pred_series.index)

print(f"\n🔀 Aligning predictions:")
print(f"   - Common prediction index: {len(common_pred_index)} timestamps")

# Create aligned DataFrame for ensemble averaging
aligned_predictions = pd.DataFrame()
for model_name, pred_series in model_predictions.items():
    aligned_predictions[model_name] = pred_series.loc[common_pred_index]

# Calculate ensemble as mean of aligned predictions
prob_up_ensemble = aligned_predictions.mean(axis=1).values

print(f"   - Ensemble probabilities: {len(prob_up_ensemble)} values")

# Create output dataframe with predictions
PREDICTIONS_15M_DF = DF_PROCESSED_15m.loc[common_pred_index].copy()
PREDICTIONS_15M_DF['Pred_Prob_Up'] = prob_up_ensemble
PREDICTIONS_15M_DF['True_Target'] = PREDICTIONS_15M_DF['target']

# Add individual model predictions
for i, (model_name, pred_series) in enumerate(model_predictions.items(), 1):
    PREDICTIONS_15M_DF[f'Prob_Model_{i}'] = pred_series.loc[common_pred_index].values

print(f"\n✅ PREDICTIONS_15M_DF created: {PREDICTIONS_15M_DF.shape}")
print(f"\n📊 Ensemble Prediction Statistics:")
print(f"   - Ensemble prob range: [{prob_up_ensemble.min():.4f}, {prob_up_ensemble.max():.4f}]")
print(f"   - Mean ensemble prob: {prob_up_ensemble.mean():.4f}")
print(f"   - Std ensemble prob: {prob_up_ensemble.std():.4f}")

print("\n" + "="*70)

🚀 BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS

📊 Top 3 models selected by forward accuracy:
   1. AdaBoost_Full - Acc: 0.5394, Features: Full
   2. LSTM_Full - Acc: 0.5377, Features: Full
   3. LSTM_Attention_Full - Acc: 0.5370, Features: Full

📋 Feature matrices created:
   - X_full (28 features): (5712, 28)
   - X_selected (18 features): (5712, 18)
   - Common index: 5712 rows
✅ AdaBoost_Full: full (28 features) | predictions: 5712 | mean prob: 0.5047

🔹 LSTM_Full: Detected as LSTM/Neural Network model
   - Reshaping for LSTM: (5712, 28) → (5712, 1, 28)
   ⚠️ LSTM prediction failed: Exception encountered when calling Sequential.call().

Cannot take the length of shape with unknown rank.

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=<unknown>, dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>
   - Attempting fallback flatten prediction...
✅ LSTM_Full: full (28 features) | predictions: 5712 | mean prob: 0.5138

🔹 LSTM_Attenti

In [16]:
# ============================================================================
# Summary Statistics for Predicted Probabilities
# ============================================================================

# Check if the predictions dataframe exists
if 'PREDICTIONS_15M_DF' in globals() and isinstance(PREDICTIONS_15M_DF, pd.DataFrame) and not PREDICTIONS_15M_DF.empty:
    # Extract the probabilities
    prob_up = PREDICTIONS_15M_DF['Pred_Prob_Up']
    
    # Calculate the range of probabilities
    prob_range = (prob_up.min(), prob_up.max())
    
    # Calculate the number of rows
    num_rows = prob_up.shape[0]
    
    # Calculate the mean and standard deviation
    prob_mean = prob_up.mean()
    prob_std = prob_up.std()
    
    # Display the summary statistics
    print("📊 Summary Statistics for Predicted Probabilities:")
    print(f"  - Range: {prob_range}")
    print(f"  - Number of Rows: {num_rows}")
    print(f"  - Mean Probability: {prob_mean:.4f}")
    print(f"  - Standard Deviation: {prob_std:.4f}")
else:
    print("❌ 'PREDICTIONS_15M_DF' is not available or is empty.")

📊 Summary Statistics for Predicted Probabilities:
  - Range: (np.float64(0.385039242156801), np.float64(0.6501619827873086))
  - Number of Rows: 5712
  - Mean Probability: 0.5095
  - Standard Deviation: 0.0558


In [ ]:
# ============================================================================
# STRATEGY WITH STOP LOSS - PROPER UNITS TRACKING
# ============================================================================

def run_strategy_with_stop_loss(df, buy_threshold=0.6, sell_threshold=0.2, 
                                  stop_loss_pct=0.01, initial_balance=1000, 
                                  trading_cost=0.001):
    """
    Run trading strategy with proper units tracking.
    
    Args:
        df: DataFrame with 'Close', 'Low', 'prob_up' columns
        buy_threshold: Probability threshold to trigger buy
        sell_threshold: Probability threshold to trigger sell
        stop_loss_pct: Stop loss percentage (e.g., 0.01 = 1%)
        initial_balance: Starting balance in USD
        trading_cost: Trading cost percentage (e.g., 0.001 = 0.1%)
    
    Returns:
        final_balance: Final balance after all trades
        trades: List of trade dictionaries
    """
    balance = initial_balance
    units = 0  # Number of units owned - CRITICAL FIX
    position = 0
    entry_price = 0
    trades = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        current_price = row['Close']
        prob = row['prob_up']
        low_price = row['Low']
        
        if position == 1:
            stop_price = entry_price * (1 - stop_loss_pct)
            
            # PRIORITY 1: Check stop loss first (using Low price for intrabar check)
            if low_price <= stop_price:
                exit_price = stop_price
                proceeds = units * exit_price * (1 - trading_cost)  # FIXED: Use units
                trades.append({
                    'type': 'STOP_LOSS',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
                continue
            
            # PRIORITY 2: Check sell threshold
            elif prob < sell_threshold:
                exit_price = current_price
                proceeds = units * exit_price * (1 - trading_cost)  # FIXED: Use units
                trades.append({
                    'type': 'SELL',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
        
        # Buy condition
        elif position == 0 and prob > buy_threshold:
            entry_price = current_price * (1 + trading_cost)
            units = balance / entry_price  # FIXED: Calculate units
            position = 1
            balance = 0
    
    # Close any remaining position at last price
    if position == 1:
        exit_price = df.iloc[-1]['Close']
        balance = units * exit_price * (1 - trading_cost)  # FIXED: Use units
    
    return balance, trades


# ============================================================================
# RUN STRATEGY WITH DEFAULT PARAMETERS
# ============================================================================

# Define thresholds for the probability to trigger buy and sell actions
buy_threshold = 0.64  # Buy when probability exceeds this threshold
sell_threshold = 0.386  # Sell when probability goes below this threshold
stop_loss_pct = 0.025  # 2.5% stop loss
trading_cost_pct = 0.001  # 0.1% trading cost
initial_balance = 1000.0  # Starting balance in USD

# Check if the predictions dataframe exists
if 'PREDICTIONS_15M_DF' in globals() and isinstance(PREDICTIONS_15M_DF, pd.DataFrame) and not PREDICTIONS_15M_DF.empty:
    # Prepare dataframe for strategy function
    df_strategy = PREDICTIONS_15M_DF.copy()
    df_strategy['prob_up'] = df_strategy['Pred_Prob_Up']
    
    # Run strategy with proper units tracking
    final_balance, trades = run_strategy_with_stop_loss(
        df_strategy,
        buy_threshold=buy_threshold,
        sell_threshold=sell_threshold,
        stop_loss_pct=stop_loss_pct,
        initial_balance=initial_balance,
        trading_cost=trading_cost_pct
    )
    
    # Calculate statistics
    total_return = (final_balance / initial_balance - 1) * 100
    
    # Count trades by type
    stop_loss_count = sum(1 for t in trades if t['type'] == 'STOP_LOSS')
    sell_count = sum(1 for t in trades if t['type'] == 'SELL')
    total_trades = len(trades)
    
    # Calculate win rate
    winning_trades = [t for t in trades if t['return'] > 0]
    win_rate = (len(winning_trades) / total_trades * 100) if total_trades else 0
    
    print(f"\n{'='*60}")
    print(f"✅ STRATEGY EXECUTION COMPLETE (WITH UNITS TRACKING)")
    print(f"{'='*60}")
    print(f"\n💰 FINAL RESULTS:")
    print(f"  - Initial Balance:  ${initial_balance:.2f}")
    print(f"  - Final Balance:    ${final_balance:.2f}")
    print(f"  - Total Return:     {total_return:+.2f}%")
    print(f"  - Profit/Loss:      ${final_balance - initial_balance:+.2f}")
    
    print(f"\n📊 TRADING STATISTICS:")
    print(f"  - Total Trades:     {total_trades}")
    print(f"  - Regular Exits:    {sell_count}")
    print(f"  - Stop Loss Exits:  {stop_loss_count}")
    print(f"  - Win Rate:         {win_rate:.2f}%")
    
    print(f"\n📋 TRADE HISTORY (Last 10 trades):")
    for i, trade in enumerate(trades[-10:], 1):
        print(f"  {i}. {trade['type']} | Entry: ${trade['entry']:.4f} | Exit: ${trade['exit']:.4f} | Return: {trade['return']*100:+.2f}% | Balance: ${trade['balance']:.2f}")
    
    # ============================================================================
    # PERFORMANCE METRICS - Sharpe Ratio and Maximum Drawdown
    # ============================================================================
    
    # Extract strategy returns
    strategy_returns = pd.Series([t['return'] for t in trades])
    
    if len(strategy_returns) > 0:
        # Sharpe Ratio (Risk-Free Rate assumed to be 0)
        excess_return = strategy_returns.mean()
        volatility = strategy_returns.std()
        sharpe_ratio = excess_return / volatility if volatility != 0 else 0
        
        # Maximum Drawdown (MDD)
        equity_curve = (1 + strategy_returns).cumprod()
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve - peak) / peak
        mdd = drawdown.min() * 100
        
        print(f"\n📊 PERFORMANCE METRICS:")
        print(f"  - Sharpe Ratio:     {sharpe_ratio:.2f}")
        print(f"  - Maximum Drawdown: {mdd:.2f}%")
else:
    print("❌ 'PREDICTIONS_15M_DF' is not available or is empty.")


✅ STRATEGY EXECUTION COMPLETE

💰 FINAL RESULTS:
  - Initial Balance:  $1000.00
  - Final Balance:    $1172.64
  - Total Return:     +17.26%
  - Profit/Loss:      $+172.64

📊 TRADING STATISTICS:
  - Total Entries:    7
  - Regular Exits:    2
  - Stop Loss Exits:  4
  - Win Rate:         42.86%

📋 TRADE HISTORY (Last 10 trades):
  1. BUY @ $2315.9604 (prob: 0.6441) | Entry: $2318.2764
  2. STOP LOSS @ $2257.3706 (prob: 0.6273) | Return: -2.60% | Balance: $974.02
  3. BUY @ $2137.0442 (prob: 0.6473) | Entry: $2139.1812
  4. STOP LOSS @ $2075.9258 (prob: 0.6361) | Return: -2.60% | Balance: $948.72
  5. BUY @ $1941.4506 (prob: 0.6424) | Entry: $1943.3920
  6. STOP LOSS @ $1875.2238 (prob: 0.6459) | Return: -2.60% | Balance: $924.08
  7. BUY @ $1785.1901 (prob: 0.6406) | Entry: $1786.9753
  8. SELL @ $2128.7747 (prob: 0.3853) | Return: +19.01% | Balance: $1099.73
  9. BUY @ $2013.3761 (prob: 0.6405) | Entry: $2015.3895
  10. STOP LOSS @ $1958.2738 (prob: 0.6298) | Return: -2.60% | Balance:

In [ ]:
# ============================================================================
# OPTIMIZE BUY/SELL THRESHOLDS AND STOP LOSS WITH GRID SEARCH (WITH UNITS TRACKING)
# ============================================================================
import numpy as np
import pandas as pd

if 'PREDICTIONS_15M_DF' not in globals() or PREDICTIONS_15M_DF.empty:
    raise ValueError("PREDICTIONS_15M_DF is missing. Run the predictions dataframe cell first.")

# Configuration
initial_balance = 1000.0
trading_cost = TRADING_COST_PCT if 'TRADING_COST_PCT' in globals() else 0.001

# Threshold and stop loss grids
buy_grid = np.round(np.linspace(0.50, 0.80, 16), 3)
sell_grid = np.round(np.linspace(0.20, 0.49, 15), 3)
stop_loss_grid = np.round(np.linspace(0.01, 0.05, 9), 4)  # 1% to 5% stop loss

print("📊 OPTIMIZATION PARAMETERS:")
print(f"  - Buy thresholds: {len(buy_grid)} values ({buy_grid[0]:.3f} to {buy_grid[-1]:.3f})")
print(f"  - Sell thresholds: {len(sell_grid)} values ({sell_grid[0]:.3f} to {sell_grid[-1]:.3f})")
print(f"  - Stop loss: {len(stop_loss_grid)} values ({stop_loss_grid[0]:.4f} to {stop_loss_grid[-1]:.4f})")
print(f"  - Total combinations: {len(buy_grid) * len(sell_grid) * len(stop_loss_grid)}")

# Prepare dataframe with Low prices for proper stop loss check
df_opt = PREDICTIONS_15M_DF.copy()

results = []

for buy_th in buy_grid:
    for sell_th in sell_grid:
        if sell_th >= buy_th:
            continue
        
        for sl_pct in stop_loss_grid:
            balance = initial_balance
            units = 0  # FIXED: Track units
            position = 0
            entry_price = 0

            for i in range(len(df_opt)):
                row = df_opt.iloc[i]
                prob = row['Pred_Prob_Up']
                price = row['Close']
                low_price = row['Low']

                if position == 1:
                    # Check stop loss first (priority 1) using Low price
                    stop_loss_price = entry_price * (1 - sl_pct)
                    if low_price <= stop_loss_price:
                        # Stop loss triggered - exit at stop loss price level
                        exit_price = stop_loss_price
                        proceeds = units * exit_price * (1 - trading_cost)  # FIXED
                        balance = proceeds
                        position = 0
                        units = 0
                        entry_price = 0
                        continue
                    # Check sell threshold (priority 2)
                    elif prob <= sell_th:
                        # Sell at current price
                        exit_price = price
                        proceeds = units * exit_price * (1 - trading_cost)  # FIXED
                        balance = proceeds
                        position = 0
                        units = 0
                        entry_price = 0
                
                elif position == 0 and prob >= buy_th:
                    # Buy
                    entry_price = price * (1 + trading_cost)
                    units = balance / entry_price  # FIXED: Calculate units
                    position = 1
                    balance = 0

            # If still in position, close at last price
            if position == 1:
                last_price = df_opt['Close'].iloc[-1]
                balance = units * last_price * (1 - trading_cost)  # FIXED

            total_return = (balance / initial_balance) - 1
            results.append({
                'buy_threshold': float(buy_th),
                'sell_threshold': float(sell_th),
                'stop_loss_pct': float(sl_pct),
                'final_balance': float(balance),
                'total_return': float(total_return)
            })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

best_row = results_df.iloc[0]
OPTIMAL_BUY_THRESHOLD = best_row['buy_threshold']
OPTIMAL_SELL_THRESHOLD = best_row['sell_threshold']
OPTIMAL_STOP_LOSS = best_row['stop_loss_pct']

print("\n" + "="*70)
print("✅ OPTIMIZATION COMPLETE (WITH UNITS TRACKING)")
print("="*70)
print(f"\n🎯 OPTIMAL PARAMETERS (Maximizing Total Return):")
print(f"  - Buy Threshold:   {OPTIMAL_BUY_THRESHOLD:.4f}")
print(f"  - Sell Threshold:  {OPTIMAL_SELL_THRESHOLD:.4f}")
print(f"  - Stop Loss:       {OPTIMAL_STOP_LOSS:.4f} ({OPTIMAL_STOP_LOSS*100:.2f}%)")
print(f"  - Final Balance:   ${best_row['final_balance']:.2f}")
print(f"  - Total Return:    {best_row['total_return']*100:+.2f}%")

print(f"\n📊 TOP 10 PARAMETER COMBINATIONS:")
display(results_df[['buy_threshold', 'sell_threshold', 'stop_loss_pct', 'final_balance', 'total_return']].head(10))

# Create visualization: Heatmaps for best stop loss configurations
print(f"\n📈 ANALYSIS BY STOP LOSS LEVEL:")
for sl in sorted(results_df['stop_loss_pct'].unique())[:3]:  # Show top 3 stop loss levels
    subset = results_df[results_df['stop_loss_pct'] == sl].nlargest(1, 'total_return')
    if not subset.empty:
        row = subset.iloc[0]
        print(f"  - SL {sl*100:.2f}%: Buy={row['buy_threshold']:.3f}, Sell={row['sell_threshold']:.3f}, Return={row['total_return']*100:+.2f}%")

📊 OPTIMIZATION PARAMETERS:
  - Buy thresholds: 16 values (0.500 to 0.800)
  - Sell thresholds: 15 values (0.200 to 0.490)
  - Stop loss: 9 values (0.0100 to 0.0500)
  - Total combinations: 2160

✅ OPTIMIZATION COMPLETE

🎯 OPTIMAL PARAMETERS (Maximizing Total Return):
  - Buy Threshold:   0.6400
  - Sell Threshold:  0.3860
  - Stop Loss:       0.0250 (2.50%)
  - Final Balance:   $1136.54
  - Total Return:    +13.65%

📊 TOP 10 PARAMETER COMBINATIONS:


,buy_threshold,sell_threshold,stop_loss_pct,final_balance,total_return
1029,0.64,0.386,0.025,1136.535018,0.136535
1028,0.64,0.386,0.020,1124.454097,0.124454
1030,0.64,0.386,0.030,1117.551388,0.117551
1063,0.64,0.469,0.015,1109.127057,0.109127
1054,0.64,0.449,0.015,1104.372156,0.104372
1031,0.64,0.386,0.035,1103.869142,0.103869
1067,0.64,0.469,0.035,1094.359352,0.094359
1058,0.64,0.449,0.035,1089.667762,0.089668
1066,0.64,0.469,0.030,1087.718992,0.087719
1064,0.64,0.469,0.020,1087.515618,0.087516



📈 ANALYSIS BY STOP LOSS LEVEL:
  - SL 1.00%: Buy=0.640, Sell=0.469, Return=+5.71%
  - SL 1.50%: Buy=0.640, Sell=0.469, Return=+10.91%
  - SL 2.00%: Buy=0.640, Sell=0.386, Return=+12.45%
